<h2>Conduct feature crossing by combining multiple variables to create new predictive attributes that capture hidden patterns, followed by assessing their contribution through cross-validation and feature selection methods.</h2>

In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(42)

In [3]:
df = pd.DataFrame({
    'house_age': np.random.randint(1, 30, 200),
    'rooms': np.random.randint(1, 6, 200),
    'location_code': np.random.choice(['A', 'B'], 200),
    'house_price': np.random.randint(100000, 500000, 200)
})

In [4]:
print(df.head())

   house_age  rooms location_code  house_price
0          7      3             B       193008
1         20      4             A       456441
2         29      2             A       125351
3         15      2             B       188668
4         11      5             A       214585


In [5]:
df['rooms_x_age'] = df['rooms'] * df['house_age']

In [6]:
df['loc_and_rooms'] = df['location_code'] + "_rooms_" + df['rooms'].astype(str)

In [7]:
df_encoded = pd.get_dummies(df, columns=['location_code', 'loc_and_rooms'], drop_first=True)

In [8]:
print(df_encoded.head())

   house_age  rooms  house_price  rooms_x_age  location_code_B  \
0          7      3       193008           21             True   
1         20      4       456441           80            False   
2         29      2       125351           58            False   
3         15      2       188668           30             True   
4         11      5       214585           55            False   

   loc_and_rooms_A_rooms_2  loc_and_rooms_A_rooms_3  loc_and_rooms_A_rooms_4  \
0                    False                    False                    False   
1                    False                    False                     True   
2                     True                    False                    False   
3                    False                    False                    False   
4                    False                    False                    False   

   loc_and_rooms_A_rooms_5  loc_and_rooms_B_rooms_1  loc_and_rooms_B_rooms_2  \
0                    False                

In [9]:
from sklearn.feature_selection import SelectKBest, f_regression

In [10]:
X = df_encoded.drop(columns=['house_price'])
y = df_encoded['house_price']

In [11]:
selector = SelectKBest(score_func=f_regression, k=3)

In [12]:
X_new = selector.fit_transform(X, y)

In [13]:
selected_features = X.columns[selector.get_support()]

In [14]:
print("Top 3 chosen features:")
print(list(selected_features))

Top 3 chosen features:
['house_age', 'location_code_B', 'loc_and_rooms_B_rooms_3']


In [15]:
from sklearn.linear_model import LinearRegression

In [16]:
from sklearn.model_selection import cross_val_score

In [17]:
model = LinearRegression()

In [18]:
scores = cross_val_score(model, X_new, y, cv=5)

In [19]:
print("Scores for each fold:", scores)

Scores for each fold: [-0.10513834  0.06849021  0.1064381  -0.07528909 -0.03050681]


In [20]:
print("Average CV Score:", scores.mean())

Average CV Score: -0.007201186975044282
